# სიტყვა: ქართული გრამატიკის AI მოდელი
ეს notebook ქმნის მონაცემებს, ადარებს baseline-სა და სამ ML მოდელს, აჩვენებს შეფასებას და უშვებს Gradio აპს.

In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn gradio joblib

In [ ]:
import re, random, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

random.seed(42)
seeds = [
'მოსწავლე ყოველ დილით ახალ წიგნს კითხულობს.', 'მასწავლებელმა რთული წესი მარტივი მაგალითით ახსნა.',
'როცა გაკვეთილი დასრულდა, ბავშვები ბიბლიოთეკაში წავიდნენ.', 'ქართული ანბანი ოცდაცამეტი ასოსგან შედგება.',
'მიუხედავად წვიმისა, მოგზაურობა მაინც გავაგრძელეთ.', 'რა თქმა უნდა, ამ საკითხს კიდევ ერთხელ განვიხილავთ.',
'ალბათ, საღამოს თეატრში ახალი სპექტაკლი იქნება.', 'რამდენიმე მოსწავლემ საინტერესო შეკითხვა დასვა.',
'ერთ-ერთი მოთხრობა განსაკუთრებით დამამახსოვრდა.', 'მე ამ მნიშვნელოვან განსხვავებას აუცილებლად აღვნიშნავ.',
'ხვალ მეგობარს ვნახავ და წიგნს დავუბრუნებ.', 'დავალებას ყურადღებით წავიკითხავ და პასუხს დავწერ.',
'ნუ იჩქარებ, ჯერ წინადადების აზრი გაიგე.', 'სამი წიგნი მაგიდაზე მოწესრიგებულად ეწყო.',
'კაცმა წერილი დაწერა და მეგობარს გაუგზავნა.', 'ჩვენი სკოლის ბიბლიოთეკაში ბევრი საინტერესო წიგნია.',
'ნელ-ნელა ცაზე ღრუბლები გაიფანტა.', 'გამარჯობა, როგორ ხარ?',
'დღეს წავიკითხეთ ლექსი, მოთხრობა და მცირე ესე.', 'ქართული ენის შესწავლა ყოველდღიურ პრაქტიკას მოითხოვს.',
'ლექსის რიტმი და მხატვრული სახეები სათქმელს აძლიერებს.', 'მოკლე პასუხიც შეიძლება ზუსტი და დასაბუთებული იყოს.',
'გაკვეთილის შემდეგ მოსწავლეები მასწავლებელს ესაუბრნენ.', 'მე წერილს ვწერ, ჩემი და კი ნახატს ხატავს.',
'სწორი სასვენი ნიშანი წინადადების აზრს უფრო ნათელს ხდის.', 'კითხვის დასმა სწავლის პროცესის ბუნებრივი ნაწილია.'
]
wrong_forms = {'რა თქმა უნდა':'რათქმაუნდა','მიუხედავად':'მიუხედავათ','ალბათ':'ალბად','რამდენიმე':'რამოდენიმე','ერთ-ერთი':'ერთერთი','აღვნიშნავ':'ავღნიშნავ','ვნახავ':'ვნახამ','დავწერ':'დავწერავ'}

def variants(text):
    out=[]
    spaces=[m.start() for m in re.finditer(' ', text)]
    if spaces:
        p=spaces[len(spaces)//2]; out.append(('double_space', text[:p]+'  '+text[p+1:]))
    m=re.search(r'[,.;:!?]', text)
    if m: out.append(('space_before_punctuation', text[:m.start()]+' '+text[m.start():]))
    m=re.search(r'([,;:!?]) ', text)
    if m: out.append(('missing_space_after_punctuation', text[:m.end()-1]+text[m.end():]))
    for good,bad in wrong_forms.items():
        if good in text: out.append(('common_spelling', text.replace(good,bad,1))); break
    for i,ch in enumerate(text):
        if i>len(text)//3 and ch in {'ა':'a','ე':'e','ო':'o'}:
            out.append(('mixed_alphabet', text[:i]+{'ა':'a','ე':'e','ო':'o'}[ch]+text[i+1:])); break
    return out

rows=[]
for i,text in enumerate(seeds):
    rows.append({'source_id':i,'text':text,'label':'correct','error_type':'none','correct_text':text})
    for kind,bad in variants(text): rows.append({'source_id':i,'text':bad,'label':'error','error_type':kind,'correct_text':text})
df=pd.DataFrame(rows)
print(df.shape); display(df['label'].value_counts()); display(df.sample(5, random_state=42))

In [ ]:
wrong_tokens=tuple(wrong_forms.values())
def grammar_features(texts):
    data=[]
    for text in texts:
        data.append([len(re.findall(r'[ \\t]{2,}',text)),len(re.findall(r'\\s+[,.;:!?]',text)),len(re.findall(r'[,;:!?](?=[^\\s\\d\\n])',text)),len(re.findall(r'[A-Za-zА-Яа-я]',text)),sum(text.count(w) for w in wrong_tokens)])
    return np.asarray(data,dtype=float)

def pipeline(model):
    features=FeatureUnion([('chars',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),sublinear_tf=True)),('rules',FunctionTransformer(grammar_features,validate=False))])
    return Pipeline([('features',features),('classifier',model)])

split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42)
train_idx,test_idx=next(split.split(df,df.label,groups=df.source_id))
train,test=df.iloc[train_idx],df.iloc[test_idx]
baseline=DummyClassifier(strategy='most_frequent').fit([[0]]*len(train),train.label)
results=[{'model':'Majority baseline','accuracy':accuracy_score(test.label,baseline.predict([[0]]*len(test))),'macro_f1':f1_score(test.label,baseline.predict([[0]]*len(test)),average='macro')}]
models={'Logistic Regression':pipeline(LogisticRegression(max_iter=2000,class_weight='balanced')),'Linear SVM':pipeline(LinearSVC(class_weight='balanced',random_state=42)),'Naive Bayes':pipeline(MultinomialNB(alpha=.35))}
predictions={}
for name,model in models.items():
    model.fit(train.text,train.label); pred=model.predict(test.text); predictions[name]=pred
    results.append({'model':name,'accuracy':accuracy_score(test.label,pred),'macro_f1':f1_score(test.label,pred,average='macro')})
comparison=pd.DataFrame(results).sort_values('macro_f1',ascending=False)
display(comparison)
best_name=comparison.iloc[0].model; best=models[best_name]; best_pred=predictions[best_name]
joblib.dump(best,'grammar_classifier.joblib')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.countplot(data=df,x='error_type',ax=axes[0],color='#1677d2'); axes[0].tick_params(axis='x',rotation=35); axes[0].set_title('მონაცემთა ტიპები')
cm=confusion_matrix(test.label,best_pred,labels=['correct','error'])
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['correct','error'],yticklabels=['correct','error'],ax=axes[1]); axes[1].set_title('Confusion matrix')
plt.tight_layout(); plt.show()
examples=test[['text','label','correct_text']].copy(); examples['prediction']=best_pred; examples['correct_prediction']=examples.label==examples.prediction
display(pd.concat([examples[examples.correct_prediction].head(3),examples[~examples.correct_prediction].head(3)]))

In [ ]:
import gradio as gr
def check_georgian(text):
    label=best.predict([text])[0]
    corrected=re.sub(r'\\s+([,.;:!?])',r'\\1',text)
    corrected=re.sub(r'([,;:!?])(?=[^\\s\\d\\n])',r'\\1 ',corrected)
    corrected=re.sub(r'[ \\t]{2,}',' ',corrected)
    for good,bad in wrong_forms.items(): corrected=corrected.replace(bad,good)
    return ('შეცდომაა აღმოჩენილი' if label=='error' else 'სავარაუდოდ გამართულია'), corrected
demo=gr.Interface(check_georgian,gr.Textbox(lines=5,label='ქართული ტექსტი'),[gr.Textbox(label='მოდელის პასუხი'),gr.Textbox(label='შესწორებული ვარიანტი')],title='სიტყვა: ქართული გრამატიკის AI')
demo.launch(share=True)